In [2]:
# =============================================================================
# 01_Data_Cleaning.ipynb
# Replication Study: Card & Krueger (1994)
# "Minimum Wages and Employment: A Case Study of the Fast-Food Industry
#  in New Jersey and Pennsylvania"
# Track A: Difference-in-Differences (DID)
# =============================================================================

# %% [markdown]
# ## Step 1: Setup and Imports

# %%
import pandas as pd
import numpy as np
import os

# %% [markdown]
# ## Step 2: Load the Raw Data
#
# `public.dat` is a **whitespace-delimited fixed-width text file**.
# This is a common format in older social science datasets.
# Missing values are coded as `"."` in the original file.
#
# Variable names come from the `codebook` file included in the zip archive.
# Variables WITHOUT a trailing "2" are from **Wave 1** (Feb/Mar 1992, pre-treatment).
# Variables WITH a trailing "2" are from **Wave 2** (Nov/Dec 1992, post-treatment).

# %%
# Define variable names based on the codebook
variable_names = [
    # --- Store identifiers ---
    'sheet',        # store ID (unique identifier)
    'chain',        # chain: 1=Burger King, 2=KFC, 3=Roy Rogers, 4=Wendy's
    'co_owned',     # 1=company-owned, 0=franchisee-owned
    'state',        # 1=New Jersey (treatment), 0=Pennsylvania (control)
    'southj',       # 1=southern NJ
    'centralj',     # 1=central NJ
    'northj',       # 1=northern NJ
    'pa1',          # 1=PA suburb of Philadelphia
    'pa2',          # 1=PA Easton area
    'shore',        # 1=NJ shore area

    # --- Wave 1 (Feb/Mar 1992, BEFORE minimum wage increase) ---
    'ncalls',       # number of calls to complete interview
    'empft',        # number of full-time employees
    'emppt',        # number of part-time employees
    'nmgrs',        # number of managers/assistant managers
    'wage_st',      # starting wage ($/hour)
    'inctime',      # months to first raise
    'firstinc',     # size of first raise ($/hour)
    'bonus',        # 1=cash bonus for recruiting new workers
    'pctaff',       # % of workers affected by minimum wage increase
    'meals',        # free/discount meals: 0=free, 1=discount, 2=both, 3=none
    'open',         # hour of opening
    'hrsopen',      # number of hours open per day
    'psoda',        # price of medium soda ($)
    'pfry',         # price of small fries ($)
    'pentree',      # price of entree ($)
    'nregs',        # number of cash registers
    'nregs11',      # number of registers open at 11:00am

    # --- Wave 2 (Nov/Dec 1992, AFTER minimum wage increase) ---
    'type2',        # type of 2nd interview: 1=phone, 2=personal
    'status2',      # status of store: 1=open, 2=closed permanently, etc.
    'date2',        # date of 2nd interview (MMDDYY)
    'ncalls2',      # number of calls to complete 2nd interview
    'empft2',       # full-time employees (Wave 2)
    'emppt2',       # part-time employees (Wave 2)
    'nmgrs2',       # managers (Wave 2)
    'wage_st2',     # starting wage (Wave 2)
    'inctime2',     # months to first raise (Wave 2)
    'firstinc2',    # size of first raise (Wave 2)
    'special2',     # 1=special recruiting incentives (Wave 2)
    'meals2',       # meals policy (Wave 2)
    'open2r',       # hour of opening (Wave 2)
    'hrsopen2',     # hours open per day (Wave 2)
    'psoda2',       # price of medium soda (Wave 2)
    'pfry2',        # price of small fries (Wave 2)
    'pentree2',     # price of entree (Wave 2)
    'nregs2',       # number of registers (Wave 2)
    'nregs112',     # registers open at 11:00am (Wave 2)
]

print(f"Total variables defined: {len(variable_names)}")

# %%
# Read public.dat from GitHub raw URL
# This ensures full reproducibility — anyone can clone the repo and run this notebook
DATA_URL = "https://raw.githubusercontent.com/ZuiTu00/Replication-card-Krueger-1994/main/data/raw/public.dat"

df = pd.read_csv(
    DATA_URL,
    sep=r'\s+',
    header=None,
    names=variable_names,
    na_values='.',
    engine='python'
)

print(f"Dataset shape: {df.shape[0]} rows (stores) x {df.shape[1]} columns (variables)")

# %%
df.head()

# %% [markdown]
# ## Step 3: Basic Data Inspection

# %%
# Check data types
df.dtypes

# %%
# Missing values summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_report = pd.DataFrame({
    'n_missing': missing,
    'pct_missing': missing_pct
})
print(missing_report[missing_report['n_missing'] > 0])

# %%
# Sample sizes by group
# state=1 -> New Jersey (treatment group: minimum wage increased)
# state=0 -> Pennsylvania (control group: no change)
print("Sample size by state:")
print(df['state'].value_counts().rename({1: 'NJ (treatment)', 0: 'PA (control)'}))

# %%
# Chain distribution
chain_map = {1: 'Burger King', 2: 'KFC', 3: "Roy Rogers", 4: "Wendy's"}
print("\nSample size by chain:")
print(df['chain'].map(chain_map).value_counts())

# %% [markdown]
# ## Step 4: Construct the Key Outcome Variable
#
# Card & Krueger define **Full-Time Equivalent (FTE) employment** as:
#
# $$FTE = empft + nmgrs + 0.5 \times emppt$$
#
# Part-time workers are weighted at 0.5 of a full-time equivalent.
# This is the standard dependent variable in their DID analysis.

# %%
# FTE for Wave 1 (before) and Wave 2 (after)
df['fte'] = df['empft'] + df['nmgrs'] + 0.5 * df['emppt']
df['fte2'] = df['empft2'] + df['nmgrs2'] + 0.5 * df['emppt2']

# Change in FTE (the DID outcome of interest)
df['delta_fte'] = df['fte2'] - df['fte']

print("FTE summary statistics (Wave 1):")
print(df['fte'].describe().round(2))
print("\nFTE summary statistics (Wave 2):")
print(df['fte2'].describe().round(2))

# %% [markdown]
# ## Step 5: Quick Sanity Check — Reproduce Table 3 Means
#
# Table 3 in Card & Krueger (1994, p.778) reports:
# - NJ mean FTE (before): ~20.4
# - PA mean FTE (before): ~23.3
#
# Let's verify we can approximate these values.

# %%
print("Mean FTE by state (Wave 1):")
print(df.groupby('state')['fte'].mean().rename({1: 'NJ', 0: 'PA'}).round(2))

print("\nMean FTE by state (Wave 2):")
print(df.groupby('state')['fte2'].mean().rename({1: 'NJ', 0: 'PA'}).round(2))

# %% [markdown]
# ## Step 6: Save Cleaned Data

# %%
# Save cleaned dataset to processed folder for downstream notebooks
output_path = '../data/processed/card_krueger_1994_cleaned.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Cleaned data saved to: {output_path}")
print(f"Final shape: {df.shape}")

# %%
# Final preview
df.head()

Total variables defined: 46
Dataset shape: 410 rows (stores) x 46 columns (variables)
           n_missing  pct_missing
empft              6          1.5
emppt              4          1.0
nmgrs              6          1.5
wage_st           20          4.9
inctime           31          7.6
firstinc          43         10.5
pctaff            44         10.7
psoda              8          2.0
pfry              17          4.1
pentree           12          2.9
nregs              6          1.5
nregs11           12          2.9
ncalls2          249         60.7
empft2            12          2.9
emppt2            10          2.4
nmgrs2             6          1.5
wage_st2          21          5.1
inctime2          66         16.1
firstinc2         80         19.5
special2          18          4.4
meals2            11          2.7
open2r            11          2.7
hrsopen2          11          2.7
psoda2            22          5.4
pfry2             28          6.8
pentree2          24          

,sheet,chain,co_owned,state,southj,centralj,northj,pa1,pa2,shore,...,open2r,hrsopen2,psoda2,pfry2,pentree2,nregs2,nregs112,fte,fte2,delta_fte
0,46,1,0,0,0,0,0,1,0,0,...,6.5,16.5,1.03,NaN,0.94,4.0,4.0,40.50,24.0,-16.50
1,49,2,0,0,0,0,0,1,0,0,...,10.0,13.0,1.01,0.89,2.35,4.0,4.0,13.75,11.5,-2.25
2,506,2,1,0,0,0,0,1,0,0,...,11.0,11.0,0.95,0.74,2.33,4.0,3.0,8.50,10.5,2.00
3,56,4,1,0,0,0,0,1,0,0,...,10.0,12.0,0.92,0.79,0.87,2.0,2.0,34.00,20.0,-14.00
4,61,4,1,0,0,0,0,1,0,0,...,10.0,12.0,1.01,0.84,0.95,2.0,2.0,24.00,35.5,11.50
